# Trabajo Práctico 2 - Grupo 02

### Modelo XGBoost

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen


El objetivo es entrenar un modelo de XGBoost sobre dos representaciones de texto (TF-IDF y BoW) con búsqueda de hiperparámetros, comparando su desempeño contra el baseline de Naive Bayes (~ 0.66 F1-macro en Kaggle) y contra el de Random Forest (~ 0.65 F1-macro en Kaggle).

XGBoost (eXtreme Gradient Boosting) es una técnica de aprendizaje por ensambles (ensemble learning), específicamente de tipo boosting. Combina múltiples modelos predictivos básicos (conocidos como "aprendices débiles"), generalmente árboles de decisión, para crear un modelo final altamente preciso y robusto.

## Importación e instalación de dependencias

In [ ]:
!pip install -q spacy
!python -m spacy download es_core_news_sm
!pip install xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 61.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight

In [ ]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [ ]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [ ]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N1: XGBoost + TF-IDF



In [ ]:
# Agrupa vectorizador y modelo en un solo objeto
pipe = Pipeline([
    ("tfidf", make_tfidf()),
    ("xgb",    XGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))])
weights = compute_sample_weight("balanced", y_train)

pipe.fit(X_train, y_train, xgb__sample_weight=weights)
y_pred = pipe.predict(X_val)
evaluate("xgb_tfidf_v1", y_val, y_pred, hyperparams={"n_estimators": 100, "vectorizer": "TF-IDF", "class_weight": "balanced"})

# Reentrenar en train completo y generar submission
weights = compute_sample_weight("balanced", np.concatenate([y_train, y_val]))
pipe.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]), xgb__sample_weight=weights)
save_model(pipe, "xgb_tfidf_v1")
make_submission(pipe, pd.DataFrame({"id": test["id"], "text": X_test}), "submission_xgb_tfidf_v1.csv");


=== xgb_tfidf_v1 ===
Hiperparámetros: {'n_estimators': 100, 'vectorizer': 'TF-IDF', 'class_weight': 'balanced'}

F1-macro:  0.6445
Precision: 0.6486
Recall:    0.6473
Accuracy:  0.6775

              precision    recall  f1-score   support

    negativa     0.7723    0.7074    0.7384      4080
      neutra     0.3786    0.4966    0.4296      2040
    positiva     0.7951    0.7380    0.7655      4080

    accuracy                         0.6775     10200
   macro avg     0.6486    0.6473    0.6445     10200
weighted avg     0.7027    0.6775    0.6875     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      2886     898       296
neutra         547    1013       480
positiva       304     765      3011
Modelo guardado → models/xgb_tfidf_v1.joblib
Predicción guardada → submissions/submission_xgb_tfidf_v1.csv  (8500 predicciones)
Distribución: clase 0: 37.2%, clase 1: 25.7%, clase 2: 37.1%
